# MicroFlow-32 ablation notebook

This notebook is a copy of `CNN_training.ipynb` prepared for the 32-feature MicroFlow extractor ablation.
The original 64-feature notebook is left unchanged. Run this notebook from top to bottom when comparing latency/accuracy/resource trade-offs.


In [ ]:
from __future__ import print_function
from matplotlib import pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical


In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

In [ ]:
  import os
  os.chdir('/home/g00n3r/projects/esp32_cl_har')

  column_names = ['user-id', 'activity', 'timestamp', 'x-axis', 'y-axis', 'z-axis']
  df = pd.read_csv("data/WISDM_ar_v1.1/WISDM_ar_v1.1_raw.txt",
                   header=None,
                   names=column_names,
                   on_bad_lines='skip')
  df['z-axis'] = df['z-axis'].replace(regex=True, to_replace=r';', value=r'')
  df['z-axis'] = df['z-axis'].apply(lambda x: float(str(x).replace(',', '.')))
  df.dropna(axis=0, how='any', inplace=True)
  df.head(20)

In [ ]:
print('Number of columns in the dataframe: %i' % (df.shape[1]))
print('Number of rows in the dataframe: %i\n' % (df.shape[0]))

In [ ]:
# Show how many training examples exist for each of the six activities
df['activity'].value_counts().plot(kind='bar',
                                   title='Training Examples by Activity Type')
plt.show()
# Better understand how the recordings are spread across the different
# users who participated in the study
df['user-id'].value_counts().plot(kind='bar',
                                  title='Training Examples by User')
plt.show()

In [ ]:
import seaborn as sns
sns.set_theme(style="ticks", palette="pastel")

# Draw a nested boxplot to show bills by day and time
sns.boxplot(data=df, x="activity", y="x-axis")
sns.despine(offset=10, trim=True)

## Розподіл прискорення по осі X для кожного типу активності

Boxplot відображає розподіл значень акселерометра (вісь X) по 6 класах активності з датасету WISDM.

- **Медіана** — горизонтальна лінія всередині box
- **IQR (25–75%)** — висота box
- **Вуса** — розкид без викидів

> Jogging і Upstairs мають найбільший розкид — характерно для динамічних рухів. Walking, Downstairs, Standing, Sitting — більш стабільні.


In [ ]:
standing = df[df['activity'] == 'Standing']

Q1 = standing['x-axis'].quantile(0.25)
Q3 = standing['x-axis'].quantile(0.75)
IQR = Q3 - Q1

outliers = standing[
  (standing['x-axis'] < Q1 - 1.5 * IQR) |
  (standing['x-axis'] > Q3 + 1.5 * IQR)
]

print(f"Кількість викидів: {len(outliers)}")
print(outliers[['user-id', 'activity', 'x-axis']].describe())
outliers

In [ ]:
ACTIVITY_ORDER = ['Walking', 'Jogging', 'Upstairs', 'Downstairs', 'Sitting', 'Standing']
FEATURE_COLUMNS = ['x', 'y', 'z']
WINDOW_SIZE = 80
STEP_SIZE = 40  # 50% overlap
TIMESTAMP_GAP_FACTOR = 3.0

def prepare_dataframe(dataframe):
    df_model = dataframe.copy().rename(
        columns={
            'user-id': 'user_id',
            'x-axis': 'x',
            'y-axis': 'y',
            'z-axis': 'z',
        }
    )

    df_model = df_model[df_model['activity'].isin(ACTIVITY_ORDER)].copy()
    df_model['user_id'] = pd.to_numeric(df_model['user_id'], errors='coerce')
    df_model['timestamp'] = pd.to_numeric(df_model['timestamp'], errors='coerce')
    df_model[FEATURE_COLUMNS] = df_model[FEATURE_COLUMNS].apply(pd.to_numeric, errors='coerce')

    df_model = df_model.dropna(subset=['user_id', 'timestamp', *FEATURE_COLUMNS]).copy()
    df_model['user_id'] = df_model['user_id'].astype(np.int32)
    df_model['timestamp'] = df_model['timestamp'].astype(np.int64)
    df_model = df_model.sort_values(['user_id', 'activity', 'timestamp']).reset_index(drop=True)

    activity_to_label = {activity: idx for idx, activity in enumerate(ACTIVITY_ORDER)}
    label_to_activity = {idx: activity for activity, idx in activity_to_label.items()}
    df_model['label'] = df_model['activity'].map(activity_to_label).astype(np.int32)

    return df_model, activity_to_label, label_to_activity

def add_contiguous_segments(dataframe, timestamp_gap_factor=TIMESTAMP_GAP_FACTOR):
    segmented_groups = []

    for (_, _), group in dataframe.groupby(['user_id', 'activity'], sort=False):
        group = group.sort_values('timestamp').copy()
        deltas = group['timestamp'].diff()
        positive_deltas = deltas[deltas > 0]

        if positive_deltas.empty:
            gap_threshold = np.inf
        else:
            gap_threshold = positive_deltas.median() * timestamp_gap_factor

        new_segment = deltas.isna() | (deltas <= 0) | (deltas > gap_threshold)
        group['segment_id'] = new_segment.cumsum().astype(np.int32)
        segmented_groups.append(group)

    return pd.concat(segmented_groups, ignore_index=True)

df_model, activity_to_label, label_to_activity = prepare_dataframe(df)
df_model = add_contiguous_segments(df_model)

print(f"Rows after cleanup: {len(df_model):,}")
print(f"Subjects: {df_model['user_id'].nunique()}")
print(f"Activities: {sorted(df_model['activity'].unique().tolist())}")
print(f"Segments: {df_model.groupby(['user_id', 'activity', 'segment_id']).ngroups:,}")

display(df_model.groupby('activity').size().rename('rows').to_frame())
display(df_model.groupby('user_id')['activity'].nunique().describe().to_frame().T)

df_model.head()


In [ ]:
def fit_zscore_stats(dataframe, feature_columns):
    means = dataframe[feature_columns].mean()
    stds = dataframe[feature_columns].std().replace(0, 1.0)
    return means, stds

def apply_zscore_stats(dataframe, feature_columns, means, stds):
    scaled = dataframe.copy()
    scaled[feature_columns] = (scaled[feature_columns] - means) / stds
    return scaled

def create_windows_from_segments(dataframe, feature_columns, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    windows = []
    labels = []
    subject_ids = []

    grouped = dataframe.groupby(['user_id', 'activity', 'segment_id'], sort=False)

    for (user_id, activity, segment_id), group in grouped:
        values = group[feature_columns].to_numpy(dtype=np.float32)
        label = int(group['label'].iloc[0])

        if len(values) < window_size:
            continue

        for start in range(0, len(values) - window_size + 1, step_size):
            end = start + window_size
            windows.append(values[start:end])
            labels.append(label)
            subject_ids.append(user_id)

    if not windows:
        empty_X = np.empty((0, window_size, len(feature_columns)), dtype=np.float32)
        empty_y = np.empty((0,), dtype=np.int32)
        empty_subjects = np.empty((0,), dtype=np.int32)
        return empty_X, empty_y, empty_subjects

    X = np.stack(windows).astype(np.float32)
    y = np.asarray(labels, dtype=np.int32)
    subjects = np.asarray(subject_ids, dtype=np.int32)
    return X, y, subjects


In [ ]:
def build_one_fold_data(dataframe, test_subject_id, feature_columns=FEATURE_COLUMNS, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    train_df = dataframe[dataframe['user_id'] != test_subject_id].copy()
    test_df = dataframe[dataframe['user_id'] == test_subject_id].copy()

    train_means, train_stds = fit_zscore_stats(train_df, feature_columns)
    train_scaled = apply_zscore_stats(train_df, feature_columns, train_means, train_stds)
    test_scaled = apply_zscore_stats(test_df, feature_columns, train_means, train_stds)

    X_train, y_train, train_subjects = create_windows_from_segments(
        train_scaled,
        feature_columns,
        window_size=window_size,
        step_size=step_size,
    )
    X_test, y_test, test_subjects = create_windows_from_segments(
        test_scaled,
        feature_columns,
        window_size=window_size,
        step_size=step_size,
    )

    return {
        'train_df': train_scaled,
        'test_df': test_scaled,
        'train_means': train_means,
        'train_stds': train_stds,
        'X_train': X_train,
        'y_train': y_train,
        'train_subjects': train_subjects,
        'X_test': X_test,
        'y_test': y_test,
        'test_subjects': test_subjects,
        'y_train_onehot': to_categorical(y_train, num_classes=len(ACTIVITY_ORDER)),
        'y_test_onehot': to_categorical(y_test, num_classes=len(ACTIVITY_ORDER)),
    }

TEST_SUBJECT_ID = 1
fold_data = build_one_fold_data(df_model, test_subject_id=TEST_SUBJECT_ID)

X_train = fold_data['X_train']
y_train = fold_data['y_train']
train_subjects = fold_data['train_subjects']
y_train_onehot = fold_data['y_train_onehot']

X_test = fold_data['X_test']
y_test = fold_data['y_test']
test_subjects = fold_data['test_subjects']
y_test_onehot = fold_data['y_test_onehot']

print(f"Train windows: {X_train.shape[0]}")
print(f"Test windows: {X_test.shape[0]}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

print('Train z-score means:')
display(fold_data['train_means'].to_frame(name='mean').T)
print('Train z-score stds:')
display(fold_data['train_stds'].to_frame(name='std').T)


In [ ]:
print('NaN in X_train:', np.isnan(X_train).sum())
print('Inf in X_train:', np.isinf(X_train).sum())
print('NaN in X_test:', np.isnan(X_test).sum())
print('Inf in X_test:', np.isinf(X_test).sum())

assert X_train.shape[1:] == (WINDOW_SIZE, len(FEATURE_COLUMNS))
assert X_test.shape[1:] == (WINDOW_SIZE, len(FEATURE_COLUMNS))
assert np.all(train_subjects != TEST_SUBJECT_ID)
assert np.all(test_subjects == TEST_SUBJECT_ID)

train_window_counts = pd.Series(train_subjects).value_counts().sort_index()
test_window_counts = pd.Series(test_subjects).value_counts().sort_index()

print('Train windows per subject:')
display(train_window_counts.to_frame(name='n_windows'))
print('Test windows per subject:')
display(test_window_counts.to_frame(name='n_windows'))

print('Train class distribution:')
display(pd.Series(y_train).map(label_to_activity).value_counts().rename_axis('activity').to_frame('windows'))
print('Test class distribution:')
display(pd.Series(y_test).map(label_to_activity).value_counts().rename_axis('activity').to_frame('windows'))

print('y_train_onehot shape:', y_train_onehot.shape)
print('y_test_onehot shape:', y_test_onehot.shape)


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

NUM_CLASSES = len(ACTIVITY_ORDER)
INPUT_SHAPE = (WINDOW_SIZE, len(FEATURE_COLUMNS))

def build_baseline_model(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(filters=32, kernel_size=5, activation='relu', padding='same'),
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

baseline_model = build_baseline_model()
baseline_model.summary()

In [ ]:
BATCH_SIZE = 64
EPOCHS = 10

print(f"Train windows: {X_train.shape[0]}")
print(f"Test windows: {X_test.shape[0]}")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

model = build_baseline_model()

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_test, y_test_onehot),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

test_loss, test_acc = model.evaluate(X_test, y_test_onehot, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

BATCH_SIZE = 64
EPOCHS = 10

subject_ids = sorted(df_model['user_id'].unique().tolist())

fold_accuracies = []
all_y_true = []
all_y_pred = []
fold_results = []

for test_subject_id in subject_ids:
    fold_data = build_one_fold_data(df_model, test_subject_id=test_subject_id)

    X_train = fold_data['X_train']
    y_train_onehot = fold_data['y_train_onehot']
    X_test = fold_data['X_test']
    y_test = fold_data['y_test']

    if len(X_train) == 0 or len(X_test) == 0:
        print(f"Skipping subject {test_subject_id}: empty train/test windows")
        continue

    model = build_baseline_model()

    model.fit(
        X_train,
        y_train_onehot,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
    )

    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    fold_acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(fold_acc)

    all_y_true.extend(y_test.tolist())
    all_y_pred.extend(y_pred.tolist())

    fold_results.append({
        'test_subject_id': test_subject_id,
        'n_train_windows': int(X_train.shape[0]),
        'n_test_windows': int(X_test.shape[0]),
        'accuracy': float(fold_acc),
    })

    print(
        f"Subject {test_subject_id:>2} | "
        f"train={X_train.shape[0]:>5} | "
        f"test={X_test.shape[0]:>4} | "
        f"acc={fold_acc:.4f}"
    )

fold_results_df = pd.DataFrame(fold_results)

print("\nLOSO summary:")
display(fold_results_df)

print(f"Mean accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Std accuracy: {np.std(fold_accuracies):.4f}")

cm = confusion_matrix(all_y_true, all_y_pred)
cm_df = pd.DataFrame(cm, index=ACTIVITY_ORDER, columns=ACTIVITY_ORDER)

print("Confusion matrix:")
display(cm_df)

report = classification_report(
    all_y_true,
    all_y_pred,
    target_names=ACTIVITY_ORDER,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
print("Classification report:")
display(report_df)

In [ ]:
FINAL_REPRESENTATIVE_SAMPLES_PER_CLASS = 32

final_means, final_stds = fit_zscore_stats(df_model, FEATURE_COLUMNS)
df_model_final = apply_zscore_stats(df_model, FEATURE_COLUMNS, final_means, final_stds)

X_final, y_final, final_subjects = create_windows_from_segments(
    df_model_final,
    FEATURE_COLUMNS,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
)

print(f"X_final shape: {X_final.shape}")
print(f"y_final shape: {y_final.shape}")
print(f"final_subjects shape: {final_subjects.shape}")

print("Final class distribution:")
display(
    pd.Series(y_final)
    .map(label_to_activity)
    .value_counts()
    .rename_axis('activity')
    .to_frame('windows')
)

rng = np.random.default_rng(42)

representative_indices = []
for class_id in range(len(ACTIVITY_ORDER)):
    class_indices = np.where(y_final == class_id)[0]

    if len(class_indices) == 0:
        continue

    sample_count = min(FINAL_REPRESENTATIVE_SAMPLES_PER_CLASS, len(class_indices))
    sampled = rng.choice(class_indices, size=sample_count, replace=False)
    representative_indices.extend(sampled.tolist())

representative_indices = np.array(representative_indices, dtype=np.int32)
X_representative = X_final[representative_indices].astype(np.float32)
y_representative = y_final[representative_indices].astype(np.int32)

print(f"Representative shape: {X_representative.shape}")
print(f"Representative labels shape: {y_representative.shape}")

print("Representative class distribution:")
display(
    pd.Series(y_representative)
    .map(label_to_activity)
    .value_counts()
    .rename_axis('activity')
    .to_frame('samples')
)

def representative_dataset():
    for i in range(len(X_representative)):
        yield [X_representative[i:i+1]]

first_rep_sample = next(representative_dataset())[0]
print("One representative sample shape:", first_rep_sample.shape)
print("Representative dtype:", first_rep_sample.dtype)
print(f"Representative value range: [{X_representative.min():.4f}, {X_representative.max():.4f}]")

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

X_final_4d = X_final[..., np.newaxis].astype(np.float32)

MICROFLOW_INPUT_SHAPE = X_final_4d.shape[1:]
NUM_CLASSES = len(ACTIVITY_ORDER)
MICROFLOW_FEATURE_DIM = 32

print("X_final_4d shape:", X_final_4d.shape)
print("MicroFlow input shape:", MICROFLOW_INPUT_SHAPE)

def build_microflow_fullconv_models(input_shape=MICROFLOW_INPUT_SHAPE, num_classes=NUM_CLASSES, feature_dim=MICROFLOW_FEATURE_DIM):
    inputs = keras.Input(shape=input_shape, name="imu_window")

    # Collapse the 3-axis dimension early with a 2D conv so the graph stays in Conv2D land.
    x = layers.Conv2D(
        filters=32,
        kernel_size=(5, 3),
        activation="relu",
        padding="valid",
        name="conv2d_1",
    )(inputs)

    x = layers.Conv2D(
        filters=feature_dim,
        kernel_size=(3, 1),
        activation="relu",
        padding="valid",
        name="conv2d_2",
    )(x)

    # At this point the tensor is expected to be (time=74, width=1, channels=feature_dim).
    x = layers.AveragePooling2D(
        pool_size=(74, 1),
        name="avg_pool_features",
    )(x)

    features = x
    logits = layers.Conv2D(
        filters=num_classes,
        kernel_size=(1, 1),
        padding="valid",
        name="classifier_head",
    )(features)
    outputs = layers.Softmax(axis=-1, name="classifier_softmax")(logits)

    classifier_model = keras.Model(inputs=inputs, outputs=outputs, name="microflow_fullconv32_classifier")
    feature_extractor_model = keras.Model(inputs=inputs, outputs=features,
name="microflow_fullconv32_feature_extractor")

    classifier_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return classifier_model, feature_extractor_model

microflow_classifier_model, microflow_feature_extractor_model = build_microflow_fullconv_models()

print("Classifier summary:")
microflow_classifier_model.summary()

print("\nFeature extractor summary:")
microflow_feature_extractor_model.summary()

In [ ]:
MICROFLOW_FINAL_BATCH_SIZE = 64
MICROFLOW_FINAL_EPOCHS = 10

y_final_onehot = to_categorical(y_final, num_classes=len(ACTIVITY_ORDER))
y_final_onehot_4d = y_final_onehot[:, np.newaxis, np.newaxis, :].astype(np.float32)

microflow_classifier_model, microflow_feature_extractor_model = build_microflow_fullconv_models()

microflow_history = microflow_classifier_model.fit(
    X_final_4d,
    y_final_onehot_4d,
    validation_split=0.1,
    epochs=MICROFLOW_FINAL_EPOCHS,
    batch_size=MICROFLOW_FINAL_BATCH_SIZE,
    verbose=1,
)

feature_sample = microflow_feature_extractor_model.predict(X_final_4d[:1], verbose=0)

print("Feature sample shape:", feature_sample.shape)
print("Feature sample dtype:", feature_sample.dtype)
print("First 8 feature values:", feature_sample.reshape(-1)[:8])

In [ ]:
MICROFLOW_REPRESENTATIVE_SAMPLES_PER_CLASS = 32

rng = np.random.default_rng(42)

representative_indices_4d = []
for class_id in range(len(ACTIVITY_ORDER)):
    class_indices = np.where(y_final == class_id)[0]

    if len(class_indices) == 0:
        continue

    sample_count = min(MICROFLOW_REPRESENTATIVE_SAMPLES_PER_CLASS, len(class_indices))
    sampled = rng.choice(class_indices, size=sample_count, replace=False)
    representative_indices_4d.extend(sampled.tolist())

representative_indices_4d = np.array(representative_indices_4d, dtype=np.int32)

X_representative_4d = X_final_4d[representative_indices_4d].astype(np.float32)
y_representative_4d = y_final[representative_indices_4d].astype(np.int32)

print(f"X_representative_4d shape: {X_representative_4d.shape}")
print(f"y_representative_4d shape: {y_representative_4d.shape}")

display(
    pd.Series(y_representative_4d)
    .map(label_to_activity)
    .value_counts()
    .rename_axis("activity")
    .to_frame("samples")
)

def representative_dataset_4d():
    for i in range(len(X_representative_4d)):
        yield [X_representative_4d[i:i+1]]

first_rep_4d = next(representative_dataset_4d())[0]
print("One 4D representative sample shape:", first_rep_4d.shape)

In [ ]:
import json
from pathlib import Path

import tensorflow as tf

EXPORT_DIR = Path("/tmp/esp32_cl_har_artifacts")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER_NAME = "microflow_fullconv32_classifier_int8"
FEATURE_NAME = "microflow_fullconv32_feature_extractor_int8"

CLASSIFIER_TFLITE_PATH = EXPORT_DIR / f"{CLASSIFIER_NAME}.tflite"
CLASSIFIER_METADATA_PATH = EXPORT_DIR / f"{CLASSIFIER_NAME}_metadata.json"

FEATURE_TFLITE_PATH = EXPORT_DIR / f"{FEATURE_NAME}.tflite"
FEATURE_METADATA_PATH = EXPORT_DIR / f"{FEATURE_NAME}_metadata.json"

def save_tflite_metadata(
    model_name,
    tflite_model,
    metadata_path,
    extra_metadata=None,
):
    interpreter = tf.lite.Interpreter(model_content=tflite_model)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]

    metadata = {
        "model_name": model_name,
        "window_size": int(WINDOW_SIZE),
        "step_size": int(STEP_SIZE),
        "feature_columns": list(FEATURE_COLUMNS),
        "activity_order": list(ACTIVITY_ORDER),
        "input_shape": [int(x) for x in input_details["shape"]],
        "output_shape": [int(x) for x in output_details["shape"]],
        "input_dtype": str(input_details["dtype"]),
        "output_dtype": str(output_details["dtype"]),
        "input_scale": float(input_scale),
        "input_zero_point": int(input_zero_point),
        "output_scale": float(output_scale),
        "output_zero_point": int(output_zero_point),
        "zscore_means": {str(k): float(v) for k, v in final_means.to_dict().items()},
        "zscore_stds": {str(k): float(v) for k, v in final_stds.to_dict().items()},
        "representative_samples": int(len(X_representative_4d)),
        "microflow_feature_dim": int(MICROFLOW_FEATURE_DIM),
    }

    if extra_metadata:
        metadata.update(extra_metadata)

    metadata_path.write_text(json.dumps(metadata, indent=2))

    return input_details, output_details, metadata

def convert_to_int8_tflite(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_4d
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter.convert()

# Export classifier
classifier_tflite = convert_to_int8_tflite(microflow_classifier_model)
CLASSIFIER_TFLITE_PATH.write_bytes(classifier_tflite)

classifier_input_details, classifier_output_details, classifier_metadata = save_tflite_metadata(
    model_name=CLASSIFIER_NAME,
    tflite_model=classifier_tflite,
    metadata_path=CLASSIFIER_METADATA_PATH,
    extra_metadata={"artifact_role": "baseline_classifier", "ablation": "microflow32"},
)

# Export feature extractor
feature_tflite = convert_to_int8_tflite(microflow_feature_extractor_model)
FEATURE_TFLITE_PATH.write_bytes(feature_tflite)

feature_input_details, feature_output_details, feature_metadata = save_tflite_metadata(
    model_name=FEATURE_NAME,
    tflite_model=feature_tflite,
    metadata_path=FEATURE_METADATA_PATH,
    extra_metadata={"artifact_role": "cl_feature_extractor", "ablation": "microflow32"},
)

print("Classifier artifact:")
print(CLASSIFIER_TFLITE_PATH)
print(CLASSIFIER_METADATA_PATH)
print("classifier input:", classifier_input_details)
print("classifier output:", classifier_output_details)

print("\nFeature extractor artifact:")
print(FEATURE_TFLITE_PATH)
print(FEATURE_METADATA_PATH)
print("feature input:", feature_input_details)
print("feature output:", feature_output_details)

In [ ]:
def inspect_tflite_ops(tflite_path):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    print(f"Model: {tflite_path.name}")
    print("input:", interpreter.get_input_details()[0])
    print("output:", interpreter.get_output_details()[0])
    print("ops:")
    for i, op in enumerate(interpreter._get_ops_details()):
        print(i, op.get("op_name"))
    print()

inspect_tflite_ops(CLASSIFIER_TFLITE_PATH)
inspect_tflite_ops(FEATURE_TFLITE_PATH)
